# Customer Dataset Exploratory Data Analysis

## Understanding Customer Profiles and Data Quality

## Notebook Objective

The objective of this notebook is to perform exploratory data analysis on the customer dataset.

This analysis aims to:

- Understand the structure and characteristics of customer data.
- Identify data quality issues such as missing values, duplicates, and inconsistent values.
- Analyze customer demographic and behavioral patterns.
- Discover potential opportunities for feature engineering.
- Define requirements for the data cleaning and transformation phases.

This notebook focuses on understanding the dataset before applying any modifications.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.api.types import is_datetime64_dtype

In [ ]:
from src.loader import DataLoader
from src.clean import DataCleaner

In [ ]:
data = DataLoader()
cleaner = DataCleaner()
df_customers = data.load_customers()
clean = cleaner.report(df_customers)
clean

In [ ]:
df_customers.head()

In [ ]:
df_information = pd.DataFrame(clean["column_statistics"]).T

In [ ]:
df_information

## Data Information Documentation

For a detailed explanation and interpretation of the dataset information, including:

- Data types
- Missing values
- Unique values
- Duplicate values
- Data quality observations

please refer to the related documentation file:

`docs/customers_eda_findings.md`

The notebook focuses on exploration and visualization, while detailed findings and decisions are maintained in the documentation for better organization and reproducibility.

In [42]:
duplicated_rows = cleaner.find_duplicates(df_customers)
print(f"Shape of duplicated rows: {duplicated_rows.shape}")

Shape of duplicated rows: (1021, 14)


In [43]:
df_duplicated_groups =(
    
    df_customers[df_customers.duplicated(keep=False)]
    
    .sort_values(by="customer_id")
    
    .reset_index(drop=True)
    
)
print(f"Shape of duplicated rows: {df_duplicated_groups.shape}")

Shape of duplicated rows: (2013, 14)


### Duplicate Records

A duplicate group consists of multiple identical rows.

* **Original record:** The first occurrence of a duplicated row.
* **Duplicated copy:** Any subsequent identical occurrence of that row.
* **Rows involved in duplicate groups:** The total number of rows belonging to duplicate groups, including both original records and duplicated copies.

In this dataset:

* **992 original records**
* **1,021 duplicated copies**
* **2,013 rows involved in duplicate groups**

Therefore:

[
2,013 = 992 + 1,021
]

The **1,021 duplicated copies** represent the redundant records, while the **2,013 rows involved in duplicate groups** represent the complete set of records affected by exact duplication.


In [36]:
print(df_duplicated_groups.groupby("customer_id")
      .size()
      .value_counts()
      )

2    963
3     29
Name: count, dtype: int64


The dataset contains 992 customer records involved in exact duplicate groups. Among them, 963 customers have two identical records, while 29 customers have three identical records. In total, 2,013 rows belong to duplicate groups, consisting of 992 original records and 1,021 redundant duplicate copies.

In [37]:
customer_id_duplicated: pd.DataFrame = (df_customers[
    
    df_customers["customer_id"]
    .duplicated(keep=False)
    ]
    .sort_values(by="customer_id")
)
print(f"Shape of customer id duplicated {customer_id_duplicated.shape}")

Shape of customer id duplicated (3534, 14)


In [38]:
print(f"Distinct samples of customer id duplicated:\n{customer_id_duplicated.nunique()}")

Distinct samples of customer id duplicated:
customer_id     1734
first_name      1142
last_name       1399
email           1693
phone_number    1727
gender             3
dob             1654
signup_date     1440
address         1734
city            1621
state             50
country          241
device_id(s)    1734
source             3
dtype: int64


In [39]:
customer_id_consistency: pd.DataFrame = (
    
    customer_id_duplicated.groupby(
        "customer_id"
    )
    .nunique()
)
print(f"Shape of customer id consistency {customer_id_consistency.shape}")

Shape of customer id consistency (1734, 13)


In [40]:
conflicting_customer_id: pd.DataFrame = (
    
    customer_id_consistency[
        customer_id_consistency
        .gt(1) # greater than 1
        .any(axis=1)
    ]
)
print(f"shape of conflicting customer id {conflicting_customer_id.shape}")

shape of conflicting customer id (768, 13)


In [41]:
conflicting_counts = (conflicting_customer_id
.gt(1)
.sum()
.sort_values(ascending=False)
)
for column, count in conflicting_counts.items():
    if count>0:
        print(f"{column}: {count}")

first_name: 757
last_name: 757


Duplicate Customer ID Investigation: 1,734 customer IDs appear in multiple records, representing 3,534 rows. Among these duplicated customer IDs, 768 have inconsistent attribute values. The inconsistencies are concentrated in first_name and last_name, with 757 duplicated customer IDs showing conflicts in each of these fields. The remaining customer attributes are consistent within the identified conflicting groups.

In [46]:
conflicting_ids = conflicting_customer_id.index

conflicting_records = (
    df_customers[
        df_customers["customer_id"].isin(conflicting_ids)
    ]
    .sort_values("customer_id")
)
conflicting_records[
    [
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone_number",
        "dob",
        "signup_date",
        "address",
        "device_id(s)"
    ]
].head(30)

,customer_id,first_name,last_name,email,phone_number,dob,signup_date,address,device_id(s)
38030,00682ed4-54ac-434e-b800-a4dfa3775f10,Melissa,PECK,shared1349@hotmail.com,6610153591,2001-01-19,2024-11-20,893 Chavez Landing Suite 262,002db8ff-ed9f-48ed-ac1b-9f977421bc66
30391,00682ed4-54ac-434e-b800-a4dfa3775f10,Melissa,Peck,shared1349@hotmail.com,6610153591,2001-01-19,2024-11-20,893 Chavez Landing Suite 262,002db8ff-ed9f-48ed-ac1b-9f977421bc66
39917,0070c246-7754-46db-a55f-b14188840068,Joseph,Smith,joseph.smith734@hotmail.com,001-740-556-0991,1996-05-10,2021-05-15,2067 Allen Ways,c33a812d-fb6d-485e-be77-fc0e1526afeb
17061,0070c246-7754-46db-a55f-b14188840068,JOSEPH,sMITH,joseph.smith734@hotmail.com,001-740-556-0991,1996-05-10,2021-05-15,2067 Allen Ways,c33a812d-fb6d-485e-be77-fc0e1526afeb
34468,00c929d1-753c-49a8-af28-eb3ffe0ad087,tINA,Frost,caitlin.richardson319@hotmail.com,(142)446-4128,1996-05-01,2016-11-01,531 Michael Fords Suite 310,14c1d235-28f3-4376-ba5c-758f17d67c68
43939,00c929d1-753c-49a8-af28-eb3ffe0ad087,Tina,Frost,caitlin.richardson319@hotmail.com,(142)446-4128,1996-05-01,2016-11-01,531 Michael Fords Suite 310,14c1d235-28f3-4376-ba5c-758f17d67c68
28123,0167fd2f-c24d-4f99-a133-6678590ae380,Michael,Valdez,michael.valdez657@gmail.com,0989040312,1963-02-26,2018-12-06,2137 Myers Cliffs,530107b6-b7f9-4da1-a3c8-bb4dcf9e5cb5
32696,0167fd2f-c24d-4f99-a133-6678590ae380,Mic2hael,VALDEZ,michael.valdez657@gmail.com,0989040312,1963-02-26,2018-12-06,2137 Myers Cliffs,530107b6-b7f9-4da1-a3c8-bb4dcf9e5cb5
25598,01695a54-eb34-4506-873d-669cc875b8d0,Sean,Cook2e,sean.cooke027@yahoo.com,+1-764-716-2672,1959-12-03,2021-10-26,8290 Cynthia Fords Apt. 413,3bbbfba2-0f31-40bc-b543-f2eabff95598
26181,01695a54-eb34-4506-873d-669cc875b8d0,Sean,Cooke,sean.cooke027@yahoo.com,+1-764-716-2672,1959-12-03,2021-10-26,8290 Cynthia Fords Apt. 413,3bbbfba2-0f31-40bc-b543-f2eabff95598


In [ ]:
missing_values: dict= cleaner.find_missing_values(df_customers)
print(missing_values["Missing columns"])
print(missing_values["Missing count"])
print(missing_values["Missing percentage"])
print(missing_values["Types of Data"])

In [ ]:
def missing_email(df):
    
    info={}
    
    for column in df.columns:
        
        if column != "email":
            
            data= df[df["email"].isna()][column].value_counts()
            
            info[column] = data
        
        
    return info


print(missing_email(df_customers))

In [ ]:
def missing_email_percentage(df):
    
    output = {}
    

    for column in df.columns:


        if column != "email":


            missing_values = df.loc[
                
                df["email"].isna(),
                column
            ]
    

            counts = missing_values.value_counts()


            percentages = (counts
                           
                           .div(counts.sum())
                           
                           .mul(100)
                           
                           .round(2)
                           )
         
         
            output[column] = percentages


    return output


missing_email_percentage(df_customers)

In [ ]:
df_customers.columns[(df_customers == "").any()]

No exact empty-string values were detected in the customer dataset.

In [ ]:
def invalid_data(df) -> pd.Series:
    invalid = {}
    for col in df.columns:
        invalid[col] = (df[col].str.strip() == "").value_counts()
    
    return invalid

invalid_data(df_customers)

Empty and whitespace-only values: No exact empty strings or whitespace-only strings were detected across the 50,000 customer records and 14 columns.

In [ ]:
for column in df_customers.columns:
    invalid_test2 = df_customers[df_customers[column].str.strip() == ""]
    assert len(invalid_test2) == 0 , f"Whitespace-only values found in {column}"

In [ ]:
df_customers[["dob", "signup_date"]]
pd.to_datetime(df_customers["dob"])
pd.to_datetime(df_customers["signup_date"])

In [ ]:
print(f"The latest recorded date: {max(df_customers["dob"])}")
print(f"The earliest recorded date: {min(df_customers["dob"])}")
print("--" * 30)
print(f"The latest recorded date: {max(df_customers["signup_date"])}")
print(f"The earliest recorded date: {min(df_customers["signup_date"])}")

In [ ]:
def date_time(df: pd.DataFrame) -> dict:
    
    date = {}

    for col in df.columns:

        if is_datetime64_dtype(df[col]):

            date[col] = {
                "Latest date": df[col].max(),
                "Earliest date": df[col].min()
            }

        else:
            if df[col].astype("string").str.strip().ne("").any():

                converted = pd.to_datetime(
                    df[col],
                    errors="coerce",
                    format=r"%Y-%m-%d"
                )

                if converted.notna().all():

                    date[col] = {
                        "Latest date": converted.max(),
                        "Earliest date": converted.min()
                    }

    return date


test = date_time(df_customers)
for col, date in test.items():
    print(col, ": ", date)

In [ ]:
print(df_customers[df_customers["dob"] >= df_customers["signup_date"]].shape)

No customers were found whose signup date was earlier than or equal to their date of birth. Therefore, all customer records satisfy the basic temporal consistency rule: the signup date occurs after the date of birth.

In [ ]:
def age(
        df: pd.DataFrame,
        dob_column: str = "dob",
        signup_column: str = "signup_date") -> dict:
    
    
    dob = pd.to_datetime(
        df[dob_column],
        errors="coerce",
        format=r"%Y-%m-%d")
    
    signup_date = pd.to_datetime(
        df[signup_column],
        errors="coerce",
        format=r"%Y-%m-%d"
    )
    
    
    age_at_signup = (
        signup_date - dob
    ).dt.days / 365.25
    
    
    return {
        "age_at_signup": {
            "minimum": age_at_signup.min().round(2),
            "maximum": age_at_signup.max().round(2),
            "mean": age_at_signup.mean().round(2),
            "median": age_at_signup.median().round(2)
        }
    }
age(df_customers)

In [ ]:
dob = pd.to_datetime(df_customers["dob"], errors="coerce", format=r"%Y-%m-%d")
signed_date = pd.to_datetime(df_customers["signup_date"], errors="coerce", format=r"%Y-%m-%d")
year = 365.25
customer_age = (signed_date - dob).dt.days / year
print(int(customer_age.max()), "Year")
print(int(customer_age.min()), "Year")


In [ ]:
print(df_customers["email"].value_counts().gt(1).sum())

In [ ]:
unique_emails_per_customer = df_customers.groupby("customer_id")["email"].nunique()
unique_emails_per_customer.value_counts()

In [ ]:
unique_customer_per_emails = df_customers.groupby("email")["customer_id"].nunique()
unique_customer_per_emails.value_counts()

In [ ]:
df_customers.info()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 18))
axes = axes.flatten()

# 1. Gender distribution
sns.countplot(
    data=df_customers,
    x="gender",
    hue="gender",
    palette="Set1",
    ax=axes[0]
)
axes[0].set_title("Gender Distribution")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Number of Customers")


# 2. Age distribution
sns.histplot(
    data=df_customers,
    x=customer_age,
    bins=30,
    kde=True,
    ax=axes[1]
)
axes[1].set_title("Age Distribution at Signup")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Number of Customers")


# 3. Top 15 countries
top_countries = df_customers["country"].value_counts().head(5)

sns.barplot(
    x=top_countries.values,
    y=top_countries.index,
    ax=axes[2]
)
axes[2].set_title("Top 5 Countries by Customer Count")
axes[2].set_xlabel("Number of Customers")
axes[2].set_ylabel("Country")


# 4. Top 15 states
top_states = df_customers["state"].value_counts().head(5)

sns.barplot(
    x=top_states.values,
    y=top_states.index,
    ax=axes[3]
)
axes[3].set_title("Top 5 States by Customer Count")
axes[3].set_xlabel("Number of Customers")
axes[3].set_ylabel("State")


# 5. Top 15 cities
top_cities = df_customers["city"].value_counts().head(5)

sns.barplot(
    x=top_cities.values,
    y=top_cities.index,
    ax=axes[4]
)
axes[4].set_title("Top 5 Cities by Customer Count")
axes[4].set_xlabel("Number of Customers")
axes[4].set_ylabel("City")


# 6. Acquisition source distribution
sns.countplot(
    data=df_customers,
    x="source",
    hue="source",
    palette="Set1",
    ax=axes[5]
)
axes[5].set_title("Customer Acquisition Source")
axes[5].set_xlabel("Source")
axes[5].set_ylabel("Number of Customers")


plt.tight_layout()
plt.show()

In [ ]:
fig , axes = plt.subplots(2, 3, figsize=(20, 12))


axes = axes.flatten()


sns.countplot(data=df_customers,
              x= "gender",
              hue="gender",
              palette="Set1",
              legend=False,
              ax = axes[0])

axes[0].set_title("Gender Distribution")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Count")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

for container in axes[0].containers:
        axes[0].bar_label(container, fmt="%.0f", padding=3)
        

sns.histplot(data=df_customers,
             x=customer_age,
             ax= axes[1],
             kde= True,)

axes[1].set_title("Customer Age Distribution")
axes[1].set_xlabel("Customer Age")
axes[1].set_ylabel("Count")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

top_five_countries = df_customers["country"].value_counts().head()
sns.barplot(
            y= top_five_countries.values,
            x = top_five_countries.index,
            hue=top_five_countries.index,
            palette="Set1",
            ax=axes[2])

axes[2].set_title("Top 5 Countries")
axes[2].set_xlabel("Country")
axes[2].set_ylabel("Count")
axes[2].grid(axis="y", linestyle="--", alpha=0.7)
axes[2].tick_params(rotation=45, axis="x")


for container in axes[2].containers:
        
        axes[2].bar_label(container, padding=3, fmt="%.0f")


top_five_states = df_customers["state"].value_counts().head(5)

sns.barplot(x=top_five_states.index,
               y = top_five_states.values,
               hue=top_five_states.index,
               palette= "Set1",
               ax = axes[3])

axes[3].set_title("Top 5 States")
axes[3].set_xlabel("State")
axes[3].set_ylabel("Count")
axes[3].grid(axis="y", linestyle="--", alpha=0.7)

for container in axes[3].containers:
        axes[3].bar_label(container, padding= 3, fmt = "%.0f")


top_five_cities = df_customers["city"].value_counts().head()

axes[4].pie(top_five_cities.values,
        labels=top_five_cities.index,
        autopct="%.2f%%",
        explode=[0.1,0,0,0,0]
        )

axes[4].set_title("Top 5 Cities")
axes[4].axis("equal")



sns.countplot(data= df_customers,
              x = "source",
              hue= "source",
              palette="Set1",
              ax= axes[5])

axes[5].set_title("Source Distribution")
axes[5].set_xlabel("Source")
axes[5].set_ylabel("Count")
axes[5].grid(axis="y", linestyle="--", alpha=0.7)

for container in axes[5].containers:
        axes[5].bar_label(container, padding=3, fmt="%.0f")



plt.tight_layout()
plt.show()

In [ ]:
df_customers.columns

In [ ]:
plt.figure(figsize=(10,6))
signup: pd.Series = pd.to_datetime(df_customers["signup_date"],
                        errors="coerce",
                        format=r"%Y-%m-%d").dt.year

trend: pd.Series = signup.value_counts().sort_index(ascending=True)

plt.plot(trend.index,
         trend.values,
         color="black",
         marker="s",
         markerfacecolor="blue",
         markersize=6,
         label="Signup trend over time"
         )

plt.title("Customer Signups Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.show()